# <center >项目案例：多模态RAG系统（VLM+OCR）开发实战</center>

&emsp;&emsp;传统的`Web`应用往往采用单体架构，前端页面由后端直接渲染（如`Django Templates`、`Flask Jinja2`等）。这种方式虽然简单，但存在诸多局限：代码耦合度高、前后端开发效率低、难以支持多端（`Web`、移动端、桌面端）复用等。

&emsp;&emsp;而<font color=red>前后端分离架构</font>将整个应用拆分为两个独立的部分：

- **后端（Backend）**：专注于业务逻辑、数据处理、`API`服务
  - 提供`RESTful API`接口（如 `/api/v1/files/upload`、`/search` 等）
  - 处理`PDF`提取、向量化、存储等核心功能
  - 不关心页面如何展示，只返回`JSON`数据

- **前端（Frontend）**：专注于用户界面和交互体验
  - 调用后端`API`获取数据
  - 渲染美观的用户界面
  - 处理用户交互（点击、上传、搜索等）

&emsp;&emsp;这种架构带来的好处非常明显：**职责清晰、开发并行、技术栈独立、易于扩展**。前端开发者可以使用最适合的框架（如`React`、`Vue`），后端开发者也可以选择最擅长的语言（如`Python`、`Node.js`），两者通过标准的`HTTP API`进行通信。

&emsp;&emsp;接下来，我们将依次对项目的前后端进行详细介绍。

# 一、多模态RAG系统前端项目介绍

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510231340278.png" alt="FuFanManus Architecture" width="800">
</div>

## 1.1 前端架构设计与技术选型

&emsp;&emsp;现在让我们深入了解前端的整体架构。本项目前端采用了<font color=red>组件化 + 配置分离 + 版本隔离</font>的设计模式，这样的设计使得代码结构清晰、易于维护和扩展。

&emsp;&emsp;首先，我们来看看前端项目的目录结构：

```markdown
    frontend/
    ├── src/
    │   ├── api/
    │   │   └── config.ts          # API 端点配置（V1/V2 版本管理）
    │   ├── config.ts              # 全局配置文件
    │   └── useTheme.ts            # 主题管理 Hook
    ├── components/
    │   ├── Sidebar.tsx            # 侧边栏（导航 + 版本切换）
    │   ├── Dashboard.tsx          # 仪表盘（统计展示）
    │   ├── KnowledgeBase.tsx      # 知识库管理
    │   ├── UploadDialog.tsx       # 文件上传对话框
    │   ├── Chat.tsx               # 对话界面
    │   └── ui/                    # UI 组件库
    ├── styles/
    │   └── globals.css            # 全局样式（含主题）
    ├── App.tsx                    # 根组件
    ├── main.tsx                   # 入口文件
    ├── package.json               # 依赖配置
    ├── vite.config.ts             # Vite 构建配置
    ├── env.template               # 环境变量模板
    └── .env                       # 本地环境配置
```

&emsp;&emsp;这个结构非常清晰：`src/` 存放核心逻辑和配置，`components/` 存放所有 UI 组件，`styles/` 存放样式文件。特别要注意的是 `src/api/config.ts`，这是我们实现版本隔离的关键文件。

&emsp;&emsp;本项目目前支持两个版本并存：`V1.0`（传统`PDF`提取）和`V2.0`（`OCR`增强版）。前端优雅地实现版本隔离核心思路是：<font color=red>共享组件 + 条件渲染 + 配置分离</font>。这主要通过`src/api/config.ts`文件进行管理。

```typescript
  export const API_CONFIG = {
    // V1.0 配置：传统 PDF 提取 + VLM提取两种模式
    v1: {
      extraction: {
        uploadEndpoint: 'http://localhost:8006/api/v1/files/upload',
        methods: ['fast', 'vlm']  // 快速模式、精确模式
      },
      chunking: {
        endpoint: 'http://localhost:8001/chunk',
        methods: ['header_recursive', 'markdown_only']
      }
    },
    
    // V2.0 配置：OCR 增强版， MinerU + PaddleOCR + DeepSeek-OCR
    v2: {
      extraction: {
        uploadEndpoint: 'http://localhost:8006/api/v2/files/upload',
        methods: ['mineru', 'paddleocr', 'deepseek']  // OCR 方法
      },
      chunking: {
        endpoint: 'http://localhost:8001/chunk/v2',
        methods: ['ocr_aware', 'layout_based']  // OCR 感知切分
      }
    }
  };

  // 根据版本获取配置
  export const getAPIConfig = (isV2: boolean) => {
    return isV2 ? API_CONFIG.v2 : API_CONFIG.v1;
  };
```

&emsp;&emsp;通过这种方式，前端可以根据用户选择的版本（通过侧边栏的版本切换按钮），动态调用不同的后端`API`端点，使用不同的处理方法。<font color=red>两个版本完全独立运行，互不影响！</font>

## 1.2 配置管理与环境变量

&emsp;&emsp;本项目前端使用`Vite`的环境变量系统来管理配置。接下来我详细讲解如何配置。首先在前端根目录（`frontend/`）下，我们需要创建一个 `.env` 文件：

```bash
     # 进入前端目录
     cd /path/to/Multimodal_RAG_OCR/frontend

     # 复制模板文件
     cp env.template .env
```

&emsp;&emsp;然后编辑 `.env` 文件，配置后端服务地址：

```bash
     # 后端服务地址配置
     VITE_MILVUS_API_URL=http://localhost:8000
     VITE_CHAT_API_URL=http://localhost:8501
     VITE_EXTRACTION_API_URL=http://localhost:8006
     VITE_CHUNK_API_URL=http://localhost:8001
```

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510231348180.png" alt="FuFanManus Architecture" width="800">
</div>

## 1.3 前端环境安装与依赖管理

&emsp;&emsp;在开始安装前端项目之前，我们需要确保系统中已经安装了`Node.js`和`npm`。如果还没有安装，请参考课件最后章节"附录、本地安装 Node.js"的内容进行安装。

&emsp;&emsp;`Node.js`项目的依赖信息都记录在 `package.json` 文件中。我们只需要运行一个命令，`npm`就会自动下载和安装所有依赖包。

```bash
    # 进入前端项目目录
    cd /path/to/Multimodal_RAG_OCR/frontend

    # 安装依赖（可能需要几分钟）
    npm install
```

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510231353599.png" alt="FuFanManus Architecture" width="800">
</div>

&emsp;&emsp;执行这个命令后，`npm`会：
1. 读取 `package.json` 中的 `dependencies` 和 `devDependencies`
2. 从`npm`仓库下载所有依赖包
3. 将依赖安装到 `node_modules/` 目录
4. 生成 `package-lock.json` 锁定依赖版本

## 1.4 前端项目启动与验证


&emsp;&emsp;环境准备就绪后，启动前端项目非常简单。在前端目录下执行：

```bash
npm run dev
```

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510231353600.png" alt="FuFanManus Architecture" width="800">
</div>

&emsp;&emsp;稍等片刻，你会看到类似这样的输出：

```
    VITE v6.3.7  ready in 250 ms

    ➜  Local:   http://localhost:5173/
    ➜  Network: http://192.168.110.131:5173/
    ➜  press h + enter to show help
```

&emsp;&emsp;这表示开发服务器已成功启动！注意这几个关键信息：

- **Local**: `http://localhost:5173/` - 本地访问地址（只能在本机访问）
- **Network**: `http://192.168.110.131:5173/` - 网络访问地址（局域网内其他设备可访问）
- **端口号**: `5173` - `Vite`的默认端口（如果被占用会自动使用 5174, 5175...）

&emsp;&emsp;在浏览器中打开 `http://localhost:5173`，你应该能看到系统的登录界面或首页。

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510231355802.png" alt="FuFanManus Architecture" width="800">
</div>


&emsp;&emsp;系统默认运行在`V1.0`模式（蓝色主题）。点击侧边栏底部的"VLM模式 v1.0.0 (点击切换 OCR v2.0.0)"，应该显示蓝色主题界面，这个模式下可以使用两个提取模式：

- 快速模式（PyMuPDF4LLM）
- 精确模式（VLM）

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510231401932.png" alt="FuFanManus Architecture" width="800">
</div>

&emsp;&emsp;点击侧边栏右下角的"VLM模式 v1.0.0 (点击切换 OCR v2.0.0)"，可以切换到`OCR v2.0.0`的白色主题模式，这个模式下可以使用三个提取模式：

- MinerU（高质量 PDF 解析）
- DeepSeek-OCR（AI驱动的OCR）
- PaddleOCR-VL（视觉语言模型OCR）

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510231401933.png" alt="FuFanManus Architecture" width="800">
</div>


&emsp;&emsp;这个切换是即时生效的，而且<font color=red>切换后刷新页面，版本状态会被保留</font>（存储在浏览器的 localStorage 中）。

# 二、多模态RAG系统后端项目介绍

&emsp;&emsp;本项目的后端采用了<font color=red>微服务架构</font>设计，将不同的功能模块拆分为独立的服务，每个服务专注于特定的业务逻辑。通过`HTTP API`进行通信，共同完成从文档上传到智能问答的完整流程：

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510231420673.png" alt="Backend Architecture" width="80%">
</div>

&emsp;&emsp;接下来，我将详细介绍每个服务的功能、配置和启动方式。

## 2.1 四个核心服务及其功能

&emsp;&emsp;后端系统由以下四个微服务组成，每个服务都有其独立的端口和职责：

<style>
.center 
{
  width: auto;
  display: table;
  margin-left: auto;
  margin-right: auto;
}
</style>

<p align="center"><font face="黑体" size=4>PDF-Extract-Kit 应用的模型类型</font></p>
<div class="center">

| 服务名称 | 默认端口 | 主要功能 | 代码路径 |
|---------|---------|---------|----------|
| **PDF提取服务** | `8006` | 文档上传、PDF解析、内容提取 | `backend/Information-Extraction/unified/unified_pdf_extraction_service.py` |
| **文本切分服务** | `8001` | Markdown切分、智能分块 | `backend/Text_segmentation/markdown_chunker_api.py` |
| **Milvus API服务** | `8000` | 向量化、知识库管理、文档检索 | `backend/Database/milvus_server/milvus_api.py` |
| **对话服务** | `8501` | 知识检索、RAG问答、流式输出 | `backend/chat/kb_chat.py` |

<div>

&emsp;&emsp;其中 PDF 提取服务，在 V1.0 版本中实现的是：（路由：`/api/v1/files/upload`）

- **`fast`** - 快速模式：使用 `PyMuPDF4LLM` 快速提取文本
  - 优点：速度快，适合纯文本PDF
  - 缺点：对复杂布局支持有限
  
- **`vlm`** - 精确模式：使用 `VLM`（视觉语言模型）多模态提取
  - 优点：可以理解图片、表格、公式等复杂内容
  - 缺点：速度较慢，需要调用多模态大模型

&emsp;&emsp;在 V2.0 版本中实现的是：（路由：`/api/v2/files/upload`），MinerU、PaddleOCR、DeepSeek-OCR三种模式。

&emsp;&emsp;2025年10月16日发布的 PaddleOCR-VL 模型直接屠榜，在全球权威榜单OmniDocBench V1.5中以92.6分夺得综合性能第一，横扫文本识别、公式识别、表格理解与阅读顺序四项SOTA。

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510211049426.png" width=80%></div>

&emsp;&emsp;而紧接其后，DeepSeek 也于 2025年10月21日发布了 DeepSeek-OCR 模型，仅需7G的显存，就能完成高精度的表格、公式识别，图片语义识别，并且在多项评测指标中一举拿下SOTA成绩。

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510211101053.png" width=80%></div>

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510201346376.png" width=80%></div>

&emsp;&emsp;现代文档不仅仅是文本。它们包含多列布局、数学公式、半扫描表格、多语言文本以及分辨率奇数的图表。像 GPT-4o 或 Qwen-VL 这样的端到端模型可以解析它们，但它们速度慢、布局混乱，并且耗费 GPU 内存。所以企业环境下往往会选择更小、更紧凑的视觉模型来为解析工作提供支持。主流的企业级OCR项目应用如下：

- MinerU:[点击进入](https://github.com/opendatalab/MinerU)

&emsp;&emsp;MinerU 是由 OpenDataLab（上海人工智能实验室下属团队）发起的一个开源工具，目标是将 PDF（含扫描件、复杂版式、多栏、多表格、多公式）转换为可机读的结构化格式（如 Markdown、JSON）以便进一步下游使用。项目的定位更偏「文档内容抽取／结构化」而不仅仅是传统 OCR。其取向是“将 PDF → Markdown/JSON”这一流程，而不仅“图片 → 文字”。

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202503271440433.png" width=80%></div>

&emsp;&emsp;`MinerU`的主要工作流程分为以下几个阶段：

1. **输入**：接收`PDF` 格式文本，可以是简单的纯文本，也可以是包含双列文本、公式、表格或图等多模态`PDF`文件;
2. **文档预处理（Document Preprocessing）**：检查语言、页面大小、文件是否被扫描以及加密状态；
3. **内容解析（Content Parsing）**：
    - 局分析：区分文本、表格和图像。
    - 公式检​​测和识别：识别公式类型（内联、显示或忽略）并将其转换为 `LaTeX` 格式。
    - 表格识别：以 `HTML/LaTeX` 格式输出表格。
    - OCR：对扫描的 `PDF` 执行文本识别。
4. **内容后处理（Content Post-processing）**：修复文档解析后可能出现的问题。比如解决文本、图像、表格和公式块之间的重叠，并根据人类阅读模式重新排序内容，确保最终输出遵循自然的阅读顺序。
5. **格式转换（Format Conversion）**：以 `Markdown` 或 `JSON` 格式生成输出。
6. **输出（Output）**：高质量、结构良好的解析文档。

&emsp;&emsp;如下是`MinerU`项目官方给出的配置参考：

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202508061340184.png" alt="MinerU 配置参考" width="80%">
</div>

&emsp;&emsp;`VLM-Transformer`则是直接利用`transformers`库中的`Vision-Language`模型处理图像+文本的多模态输入，其流程如下：

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202508061356866.png" alt="MinerU 配置参考" width="800">
</div>

&emsp;&emsp;而`VLM-Sglang`后端则是利用`sglang`高性能推理引擎，优化了GPU加速及分布式部署的基础上同时支持实时流式输出，其流程如下：

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202508061356864.png" alt="MinerU 配置参考" width="800">
</div>

&emsp;&emsp;MinerU 项目非常适用于：

- 需要从大量 PDF 文档中抽取结构化内容（例如学术文献、技术白皮书、报告）用于知识库或训练语料。

- 对版式结构（如章节、列表、表格、公式）要求较高，而不只是 OCR 文本识别。

- 希望输出 Markdown／JSON 供后续自动化流水线使用。

- PaddleOCR：[点击进入](https://github.com/PaddlePaddle/PaddleOCR/tree/main)

&emsp;&emsp;PaddleOCR 是由 Baidu (及其生态) 基于其深度学习框架 PaddlePaddle 提供的开源 OCR 工具箱。支持从 PDF 或图像文档转为结构化数据（适配 AI 场景），支持 100+ 语言。最新版本 3.0 在其技术报告中提出：PP-OCRv5、PP-StructureV3、PP-ChatOCRv4 三大解决方案，覆盖文字识别、多版式文档解析、关键 信息提取。

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510211220985.png" width=80%></div>

&emsp;&emsp;在早期版本（如 PP-OCRv3）中，其结构可概括为：“检测 (Detection) → 分类 (Classification of orientation) → 识别 (Recognition)”。使用多种模型，例如检测模型（DBNet 等）、识别模型（如 SVTR），在 3.0 版本中，其“PP-StructureV3”整合了布局分析、表格识别、结构抽取。同时还最新推出了还推出了PaddleOCR-VL 的 Vision-Language 模型版本（0.9B 参数的 VLM），用于多语种文档解析。

&emsp;&emsp;PaddleOCR-VL 是推出的一个专注于“文档解析／视觉-语言模型 (Vision-Language Model, VLM)”功能的新模块，采用了视觉-语言模型架构以应对更高阶的需求。在解析多模态数据方面，PaddleOCR将这项工作分为两部分：

1. 首先检测并排序布局元素。
2. 使用紧凑的视觉语言模型精确识别每个元素。

&emsp;&emsp;该系统分为两个明确的阶段运行。

&emsp;&emsp;第一阶段是执行布局分析（PP-DocLayoutV2），此部分标识文本块、表格、公式和图表。它使用：
- RT-DETR 用于物体检测（基本上是边界框 + 类标签）。
- 指针网络 （6 个转换器层）可确定元素的读取顺序 ，从上到下、从左到右等。

&emsp;&emsp;最终输出统一模式的图片标注数据，如下图所示：

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510211239047.png" width=80%></div>

&emsp;&emsp;第二阶段则是元素识别（PaddleOCR-VL-0.9B），这就是视觉语言模型发挥作用的地方。它使用：
- NaViT 风格编码器 （来自 Keye-VL），可处理动态图像分辨率。无平铺，无拉伸。
- 一个简单的 2 层 MLP， 用于将视觉特征与语言空间对齐。
- ERNIE-4.5–0.3B 作为语言模型，该模型规模虽小但速度很快，并且采用 3D-RoPE 进行位置编码

&emsp;&emsp;最终模型输出结构化 Markdown 或 JSON 格式的文件用于后续的处理。

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510211243206.png" width=80%></div>

&emsp;&emsp;这个小小的设计决策， 将布局和识别分离，使得 PaddleOCR-VL 比通常的一体化系统更快、更稳定。同时根据实际的测试，其运行和解析速度也更快。在 A100 GPU 上， 吞吐量为 1.22 页/秒，。比 MinerU2.5 快 15.8%， VRAM 比 dots.ocr 少约 40%。

- 集成 MinerU OCR 服务

&emsp;&emsp;这里推荐大家使用Docker镜像管理MinerU项目，其地址如下：https://opendatalab.github.io/MinerU/zh/quick_start/docker_deployment/

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510231526302.png" width=80%></div>

&emsp;&emsp;首先拉取Dockerfile:

```bash
    wget https://gcore.jsdelivr.net/gh/opendatalab/MinerU@master/docker/china/Dockerfile
``` 

&emsp;&emsp;然后修改Dockerfile 内vllm镜像地址，因为默认拉取的是mineru最新可用版本，而当前最新的是2.5.4，所以这里我们将版本名定为2.5.4，方便看出拉取的代码版本，

```bash
    FROM swr.cn-north-4.myhuaweicloud.com/ddn-k8s/docker.io/vllm/vllm-openai:v0.10.1.1
```

&emsp;&emsp;构建镜像：

```bash
    docker build -t mineru-vllm:2.5.4 -f Dockerfile .
```

&emsp;&emsp;最后直接启动：

```bash
    docker compose -f compose.yaml --profile vllm-server --profile api --profile gradio up -d
```

&emsp;&emsp;启动后，可以通过 `docker ps` 查看启动的容器：

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510231531499.png" width=80%></div>

&emsp;&emsp;因为默认拉取的是mineru最新可用版本，而当前最新的是2.5.4，所以这里我们将版本名定为2.5.4，方便看出拉取的代码版本，


&emsp;&emsp;MinerU的代码集成位置：

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510231533895.png" width=80%></div>

&emsp;&emsp;而 DeepSeek-OCR 和 PaddleOCR-VL 的本地集成位置如下：



<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510231717711.png" width=80%></div>

&emsp;&emsp;同时需要说明的是，因为两个版本生成的解析数据不同，所以在构建向量数据库时的元数据也会存在一些差别：

```json
    # V1 版本
    {
      "id": "chunk_uuid",
      "vector": [0.1, 0.2, ...],  # embedding向量
      "chunk_text": "chunk内容",
      "filename": "test.pdf",
      "file_id": "file_xxx",
      "metadata": {
        "method": "fast",
        "chunk_index": 0,
        "page_start": 1,
        "page_end": 1,
        "pages": [1]
      }
    }


    # V2 版本
    {
      "id": "chunk_uuid",
      "vector": [0.1, 0.2, ...],  # embedding向量
      "chunk_text": "chunk内容",
      "filename": "test.pdf",
      "file_id": "file_xxx",
      "metadata": {
        "backend": "mineru",
        "version": "2.0",
        "extraction_method": "mineru",
        "chunk_index": 0,
        "page_start": 1,
        "page_end": 1,
        "layout_info": {...},  # V2特有：布局信息
        "has_images": true,    # V2特有：是否包含图片
        "has_page_images": true  # V2特有：是否有页面标注图
      }
    }

```

## 2.2 后端环境配置与`.env`文件说明

&emsp;&emsp;后端所有服务的配置都统一管理在项目根目录的`.env`文件中。以下是完整的后端配置说明，我们将其分为几个模块来理解：

- **模块1：生成模型（多模态）服务配置**

```bash
    # ============ 生成模型（多模态）服务配置 ============
    # 用于VLM模式和对话服务

    # 如果用OpenAI
    # MODEL_URL = "https://api.openai.com/v1"
    # API_KEY = "sk-your-openai-key"
    # MODEL_NAME = "gpt-4o"

    # 如果用Qwen（通义千问）
    API_KEY = "sk-0dbaa643a8d844cba6db6b9dc730a4db"
    MODEL_NAME = "qwen3-vl-plus"
    MODEL_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"
```

&emsp;&emsp;用于`VLM`模式的提取以及对话问答的生成。

- **模块2：向量嵌入服务配置**

```bash
    # ============ 向量嵌入服务配置 ============
    # 用于文本向量化和相似度检索

    EMBEDDING_URL=https://dashscope.aliyuncs.com/compatible-mode/v1
    EMBEDDING_MODEL_NAME=text-embedding-v4
    EMBEDDING_API_KEY=sk-0dbaa643a8d844cba6db6b9dc730a4db
```

&emsp;&emsp;用于文本向量化和相似度检索。


- **模块3：PDF上传和提取服务配置**（端口`8006`）

```bash
    # ============ PDF 上传和提取服务配置 ============

    # PDF上传文件存储路径
    UPLOAD_BASE_DIR=/home/MuyuWorkSpace/01_TrafficProject/Multimodal_RAG_OCR/backend/output/uploads

    # PDF提取结果存储路径
    EXTRACTION_RESULTS_DIR=/home/MuyuWorkSpace/01_TrafficProject/Multimodal_RAG_OCR/backend/output/extraction_results

    # 文件大小限制（MB）
    MAX_FILE_SIZE_MB=50

    # 信息抽取服务端口配置
    INFOR_EXTRAC_SERVICE_PORT=8006
    INFOR_EXTRAC_SERVICE_HOST=0.0.0.0

    # MinerU 解析后的可视化图片存储地址
    MINERU_VIZ_DIR=/home/MuyuWorkSpace/01_TrafficProject/Multimodal_RAG_OCR/backend/mineru_visualizations
```
&emsp;&emsp;其中上述配置主要用于文件上传及解析服务。

- `UPLOAD_BASE_DIR`: 上传的PDF原始文件保存目录
- `EXTRACTION_RESULTS_DIR`: 提取结果（JSON、Markdown）保存目录
- `MAX_FILE_SIZE_MB`: 单个PDF文件大小限制
- `INFOR_EXTRAC_SERVICE_PORT`: 提取服务监听端口（默认`8006`）
- `INFOR_EXTRAC_SERVICE_HOST`: 监听地址（`0.0.0.0`表示接受所有网络接口的请求）
- `MINERU_VIZ_DIR`: MinerU生成的带标注框的可视化图片存储路径

-  **模块4：文本切分服务配置**（端口`8001`）

```bash
    # 切分服务端口配置
    CHUNK_SERVICE_PORT=8001
    CHUNK_SERVICE_HOST=0.0.0.0

    # 是否启用自动切分
    CHUNKING_SERVICE_ENABLED=true
```

- **模块5：Milvus向量数据库配置**

```bash
    # ============ Milvus 向量数据库配置 ============

    # Milvus服务地址（通过Docker部署在本地）
    MILVUS_HOST=localhost
    MILVUS_PORT=19530
```

&emsp;&emsp;其中上述配置主要用于向量数据库的配置。注意：需要先通过Docker启动Milvus服务


- **模块6：Milvus API服务配置**（端口`8000`）

```bash
    # ============ Milvus API服务配置 ============

    # Milvus API服务地址
    MILVUS_API_HOST=0.0.0.0
    MILVUS_API_PORT=8000
    MILVUS_API_ENABLED=true
```

&emsp;&emsp;其中上述配置主要用于Milvus API服务的配置。


- **模块7：对话服务配置**（端口`8501`）

```bash
    # 对话服务端口配置
    CHAT_SERVICE_PORT=8501
    CHAT_SERVICE_HOST=0.0.0.0
```

## 2.3 Python虚拟环境创建与依赖安装

&emsp;&emsp;在开始使用后端服务之前，我们需要创建一个Python虚拟环境并安装项目依赖。在项目根目录下执行以下命令：

```bash
    # 进入项目根目录
    cd /path/to/Multimodal_RAG_OCR

    # 或者使用 conda（如果你使用Anaconda）
    conda create -n vlm_rag python=3.10 -y
```

&emsp;&emsp;创建虚拟环境后，需要激活虚拟环境：

**Linux/Ubuntu:**
```bash
    conda activate vlm_rag
```
<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510231733895.png" width=80%></div>

&emsp;&emsp;激活成功后，命令行提示符前会出现`(vlm_rag)`字样。项目的所有Python依赖都记录在`backend/requirements.txt`文件中。执行以下命令安装：

```bash
    # 进入后端目录
    cd backend

    # 安装依赖（可能需要几分钟）
    pip install -r requirements.txt

    # 如果下载速度慢，可以使用国内镜像源
    pip install -r requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple
```
<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510231734004.png" width=80%></div>

&emsp;&emsp;如果没有报错，说明依赖安装成功！

&emsp;&emsp;本项目提供了自动化脚本来管理所有后端服务。你可以选择一键启动所有服务，或者单独启动某个服务进行调试。一次性启动所有四个核心服务方法如下：

```bash
  # 进入后端目录
  cd /path/to/Multimodal_RAG_OCR/backend

  # 赋予执行权限（首次需要）
  chmod +x start_all_services.sh

  # 一键启动所有服务
  ./start_all_services.sh
```

&emsp;&emsp;启动后，你会看到类似以下输出：

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510231736856.png" width=80%></div>

&emsp;&emsp;完整启动前后端项目后，即可在网页中进行多模态知识库项目的运行。

<div align=center><img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510231738983.png" width=80%></div

# 附录：Windows 安装 Node.js

&emsp;&emsp;`Node.js` 是一个基于 `Chrome V8` 引擎的 `JavaScript` 运行时环境，广泛应用于前端开发、后端服务器开发等场景。接下来我们详细介绍如何在 `Windows` 系统上从零开始安装和配置 `Node.js`，包括环境变量设置、`npm` 配置等关键步骤。

- **Step 1：下载 Node.js 安装包**

&emsp;&emsp;首先需要从官方网站下载适合你系统的 `Node.js` 安装包。`Node.js` 提供了稳定的 `LTS`（长期支持）版本和最新的 `Current` 版本，推荐选择 `LTS` 版本以确保稳定性。访问 `Node.js` 官网下载页：
   - 官方地址：[https://nodejs.org/en/download/](https://nodejs.org/en/download/)
   - 中文镜像：[https://nodejs.cn/download/](https://nodejs.cn/en/download)

&emsp;&emsp;优先选择 **LTS（长期支持）版本**，稳定性更好, 选择 `.msi` 安装包（根据系统架构选择 64-bit 或 32-bit）

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202509181529316.png" alt="FuFanManus Architecture" width="800">
</div>

&emsp;&emsp;**点击下载**对应的 `.msi` 安装程序即可。

- **Step 2：运行安装程序**

&emsp;&emsp;下载完成后，需要运行 `.msi` 安装程序进行安装。安装过程中需要注意安装路径的选择以及环境变量的配置。首先找到下载的 `.msi` 文件，双击运行：

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202509181535199.png" alt="FuFanManus Architecture" width="800">
</div>

&emsp;&emsp;安装向导出现后，点击 **Next / 下一步** 继续，阅读 `License Agreement`，勾选 "I accept the terms in the License Agreement"，点击 **Next**：

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202509181535201.png" alt="FuFanManus Architecture" width="800">
</div>

&emsp;&emsp;这里强烈建议选择默认的路径：`C:\Program Files\nodejs\`，如必须修改，注意：路径中不要包含中文字符或特殊符号。

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202509181535202.png" alt="FuFanManus Architecture" width="800">
</div>

&emsp;&emsp;自定义设置， 点击 "Install" 开始安装，系统可能会弹出权限确认，选择"是"或"允许"
   - 勾选 "Add to PATH" 选项
   - 勾选 "Automatically install npm package manager"
   - 这样可以在任何命令行窗口中直接使用 `node` 和 `npm` 命令

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202509181535203.png" alt="FuFanManus Architecture" width="800">
</div>

&emsp;&emsp;等待安装完成即可。

# 附录：在 Ubuntu 系统上安装 Node.js

&emsp;&emsp;在实际的生产环境和服务器部署中，`Linux`系统（尤其是`Ubuntu`）是最常见的选择。相比`Windows`，`Linux`系统更加轻量、稳定，且对开发者友好。因此，掌握在`Ubuntu`上安装和配置`Node.js`是每一位全栈开发者的必备技能。

&emsp;&emsp;接下来，我将为大家详细讲解在`Ubuntu`系统上安装`Node.js`的方法。首先，我们需要更新系统的软件包列表，确保获取最新的软件信息：

```bash
    # 更新软件包索引
    sudo apt update

    # 升级已安装的软件包（可选）
    sudo apt upgrade -y
```

&emsp;&emsp;这个命令会联网下载最新的软件包列表，确保我们安装的是最新可用版本。接下来安装 Node.js 和 npm

```bash
    # 一条命令同时安装 Node.js 和 npm
    sudo apt install nodejs npm -y
```

&emsp;&emsp;等待安装完成后，验证安装：

```bash
    node -v    # 显示 Node.js 版本（可能是 v12 或 v14）
    npm -v     # 显示 npm 版本
```

<div align="center">
    <img src="https://muyu20241105.oss-cn-beijing.aliyuncs.com/images/202510231332604.png" alt="FuFanManus Architecture" width="800">
</div>